This helper notebook will assist with loading the wine data into pyton

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np

In [ ]:
#paths to the data
data_directory= '/content/drive/MyDrive/AMATH482HW3Data'
train_path=os.path.join(data_directory,'wine_training.csv')
test_path=os.path.join(data_directory,'wine_test.csv')

In [ ]:
#loading the data
d_train = np.loadtxt(train_path, dtype=float, delimiter=',')
d_test = np.loadtxt(test_path, dtype=float, delimiter=',')

X_train = d_train[:, 0:11]
X_test = d_test[:, 0:11]

Y_train = d_train[:,11]
Y_test = d_test[:,11]

print("Training inputs shape: "+str(X_train.shape))
print("Training outputs shape: "+str(Y_train.shape))
print("Test inputs shape: "+str(X_test.shape))
print("Test outputs shape: "+str(Y_test.shape))


Training inputs shape: (1115, 11)
Training outputs shape: (1115,)
Test inputs shape: (479, 11)
Test outputs shape: (479,)


In [ ]:
# 1. Your first step after reading in the data should be to normalize and center the training input features
# (the 𝑥𝑗 ’s) as the training outputs (the 𝑦𝑗 ’s) so that they have mean 0 and standard deviation 1.
# How should you transform the test data?
from sklearn.preprocessing import StandardScaler

x_standard_scaler = StandardScaler()
y_standard_scaler = StandardScaler()

X_train_scaled = x_standard_scaler.fit_transform(X_train)
Y_train_scaled = y_standard_scaler.fit_transform(Y_train.reshape(-1,1)).flatten()

X_test_scaled = x_standard_scaler.transform(X_test)
Y_test_scaled = y_standard_scaler.transform(Y_test.reshape(-1,1)).flatten()

In [ ]:
# 2. Fit a first-order linear model to the training data using ordinary least squares.
# What is the test MSE?
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

linear_model = LinearRegression()
linear_model.fit(X_train_scaled, Y_train_scaled)

Y_predicted = linear_model.predict(X_test_scaled)
test_MSE = mean_squared_error(Y_test_scaled, Y_predicted)

print(f"test MSE for first order linear model: {test_MSE}")


test MSE for first order linear model: 0.7471696905187211


In [ ]:
# 3. For the model in part 2, identify the indices of the five predictors with the largest (in magnitude)
# regression coefficients.
coefficients = linear_model.coef_
coefficients_sorted = np.argsort(np.abs(linear_model.coef_))[6:]

print(f"sorted coefficients: {coefficients_sorted}")

sorted coefficients: [ 4  6  9  1 10]


In [ ]:
# 4. Fit a second-order (quadratic) regression model to the data using ordinary least squares. What is the test MSE?
X_train_scaled_squared = X_train_scaled**2
X_test_scaled_squared = X_test_scaled**2

X_train_quadratic = np.hstack([X_train_scaled_squared, X_train_scaled])
X_test_quadratic = np.hstack([X_test_scaled_squared, X_test_scaled])

quadratic_model = LinearRegression()
quadratic_model.fit(X_train_quadratic, Y_train_scaled)

Y_predicted_quadratic = quadratic_model.predict(X_test_quadratic)
test_MSE_quadratic = mean_squared_error(Y_test_scaled, Y_predicted_quadratic)

print(f"test MSE for quadratic terms: {test_MSE_quadratic}")

# 5. Based on your results, does including second-order (quadratic) terms improve performance for this
# dataset? Explain your answer.
# The test MSE for the second order quadratic terms is higher than the test MSE for the regular scaled terms.
# Thus, I conclude that including second order quadratic terms does not improve the performance for this dataset.


test MSE for quadratic terms: 0.7805821192127657


In [ ]:
# 6. The list of indices from part 2 should be index=[1,4,6,9,10]. Fit a model that includes all first order
# terms and interaction terms with indices in index.
def make_interaction_terms(X, indeces):
  interaction_terms = []
  for i in indeces:
    for j in indeces:
      if i < j:
        interaction_terms.append((X[:, i] * X[:, j]).reshape(-1, 1))

  return np.hstack(interaction_terms)
X_train_interaction_terms = make_interaction_terms(X_train_scaled, coefficients_sorted)
X_test_interaction_terms = make_interaction_terms(X_test_scaled, coefficients_sorted)

X_train_all_terms = np.hstack([X_train_scaled, X_train_interaction_terms])
X_test_all_terms = np.hstack([X_test_scaled, X_test_interaction_terms])

interaction_model = LinearRegression()
interaction_model.fit(X_train_all_terms, Y_train_scaled)

Y_predicted_all_terms = interaction_model.predict(X_test_all_terms)
test_MSE_all_terms = mean_squared_error(Y_test_scaled, Y_predicted_all_terms)

# What is the test MSE?
print(f"test MSE for all terms: {test_MSE_all_terms}")

test MSE for all terms: 0.7175224728219504


In [ ]:
from sklearn.linear_model import LassoCV

# 7. Perform model selection for this interaction model using the lasso. Use cross-validation on the training
# set to select the regularization parameter. Then refit the lasso model on the full training set using the
# selected value of the regularization parameter, and evaluate its performance on the test set. Which
# value did you select, which coefficients are nonzero in the refitted model, and which model considered
# so far achieves the lowest test-set MSE?

alphas = [10**(-2), 10**(-1), 1, 10, 100]

lasso_cv = LassoCV(alphas = alphas, cv=5, max_iter = 10000)
lasso_cv.fit(X_train_all_terms, Y_train_scaled)

best_alpha = lasso_cv.alpha_
print(f"best alpha: {best_alpha}")

Y_predicted_lasso = lasso_cv.predict(X_test_all_terms)
test_MSE_lasso = mean_squared_error(Y_test_scaled, Y_predicted_lasso)
print(f"test MSE for lasso: {test_MSE_lasso}")

lasso_coefficients = lasso_cv.coef_
non_zero_coefficients = np.nonzero(lasso_coefficients)[0].tolist()
print(f"lasso non-zero coefficients: {non_zero_coefficients}")

best alpha: 0.01
test MSE for lasso: 0.7041328944186469
lasso non-zero coefficients: [1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14, 16, 18, 19]
